# View the words and labels used in the SST and agent experiments

**Run all using Zyion for both Drive prompts. No GPU needed.** This is a small metadata export and viewer, not another EEG/model experiment. It works on an existing CPU or GPU runtime without changing hardware. No package installation, EEG signals, feature preparation, classification or relabeling. New CSV outputs share a **20 MB cap**, normally only a few MB, under `Thesis/Results/word_sentiment_eeg/word_label_overview/`.

Start with **sst_vs_agent.csv**. It shows each source word occurrence, its sentence, SST class, agent lexical/context labels, how many accepted SST fixations existed, and how many survived agent selection. Look at `outcomes` and `annotation_join_status` for exclusions. The target word can appear repeatedly in different sentences with different contextual judgments.

Other lists:
- **all_agent_annotations.csv:** the complete original source-word review, including neutral and excluded cases, across the listed ZuCo tasks. This is not an accepted EEG roster.
- **sst_paper_used_words.csv:** word occurrences contributing accepted SST-labelled ZuCo SR EEG.
- **agent_paper_used_words.csv:** occurrences contributing the retained agent-labelled matched SR subset.
- **agent_fixed_sweep_used_words.csv:** the first sweep's reader-filtered agent pool, if its saved cohort is available.
- **word_types.csv:** compact spellings/labels with counts. Different labels remain separate rows.
- **experiment_index.csv:** maps the saved paper-style experiments to these pools. It explicitly distinguishes earlier first-fixation experiments, whose different accepted EEG rosters are not exported here.

These are input pools, not the precise members of every individual classifier fold or averaging repeat. Tail dropping or overlap purging can remove events for a particular fit. `accepted_fixation_trials` already sums readers and may include rereadings; do not multiply by reader count again. One source occurrence is one word location in one sentence, not necessarily a distinct spelling.

[View the original agent source CSV on GitHub](https://github.com/parmisbathayan/word-sentiment-eeg/blob/main/annotations/word_labels.csv).


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/word_label_overview'
REPO=Path('/content/word-sentiment-eeg-word-lists')
CODE_REVISION='713072dc108c1ba58f67ff622fcb2a652e9f81fc'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# Verify account quota before loading any EEG. Use Zyion for both prompts.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated quota account:',email)
if email!='zyionygvmutch39@gmail.com':
    raise RuntimeError('Choose your Zyion account for Drive authentication and mounting; it contains the saved trials.')
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a storage limit; cannot verify free space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,20_000_000-used):
    raise RuntimeError('Reserve up to 20 MB for the feature cache and results. No EEG loading has started.')
print(f'Free space: {FREE_DRIVE_BYTES/1e9:.2f} GB; new output cap: 20 MB.')
print('New output folder:',OUT)


## 1. Export the metadata tables

In [ ]:
command=[sys.executable,'-u',str(REPO/'scripts/word_label_overview.py'),
         '--thesis-root',str(THESIS),'--annotations',str(REPO/'annotations/word_labels.csv')]
subprocess.run(command,check=True)
display(pd.read_csv(OUT/'counts.csv'))
display(pd.read_csv(OUT/'experiment_index.csv'))


## 2. Search the complete table

The default table compares SST and agent selection. Change `TABLE` to another filename above if desired. Leave `SEARCH_WORD` empty for all rows. The viewer offers sorting and pagination.

In [ ]:
TABLE='sst_vs_agent.csv'
SEARCH_WORD=''
frame=pd.read_csv(OUT/TABLE,keep_default_na=False)
if SEARCH_WORD:
    column='word' if 'word' in frame else 'word_identity'
    frame=frame[frame[column].astype(str).str.contains(SEARCH_WORD,case=False,regex=False)]
print('Rows shown:',len(frame))
from google.colab import data_table
import inspect
options=dict(include_index=False,num_rows_per_page=20)
if 'max_rows' in inspect.signature(data_table.DataTable).parameters:
    options['max_rows']=max(20000,len(frame))
display(data_table.DataTable(frame,**options))


## 3. The convincing example

In [ ]:
agent=pd.read_csv(OUT/'all_agent_annotations.csv',keep_default_na=False)
example=agent[agent.word.eq('convincing') & agent.sentence.str.contains('Weiss and Speck',regex=False)]
display(example[['word','sentence','lexical_class','contextual_class','annotation_note']])
comparison=pd.read_csv(OUT/'sst_vs_agent.csv',keep_default_na=False)
accepted=comparison[comparison.word.eq('convincing') & comparison.sentence.str.contains('Weiss and Speck',regex=False)]
if len(accepted):
    display(accepted)
else:
    print('This occurrence is not in the saved accepted SST EEG pool. Its original SST word rating was independently verified positive; that does not guarantee surviving EEG quality/matching.')


## 4. Open the CSV files in Drive

Files remain private in your existing Results structure. The following cell resolves the actual Drive folder and file links; it does not upload or share anything.

In [ ]:
from IPython.display import Markdown
session=AuthorizedSession(credentials)
try:
    folder='root'
    for name in ['Thesis','Results','word_sentiment_eeg','word_label_overview']:
        response=session.get('https://www.googleapis.com/drive/v3/files',params={
            'q':f"'{folder}' in parents and name='{name}' and mimeType='application/vnd.google-apps.folder' and trashed=false",
            'fields':'files(id,name,webViewLink)'},timeout=30)
        response.raise_for_status();matches=response.json().get('files',[])
        if len(matches)!=1:
            raise RuntimeError('Folder links are not yet unambiguous in Drive. The local export is still saved.')
        found=matches[0];folder=found['id']
    if found.get('webViewLink'):
        display(Markdown(f"[Open word-list folder]({found['webViewLink']})"))
    response=session.get('https://www.googleapis.com/drive/v3/files',params={
        'q':f"'{folder}' in parents and trashed=false",'fields':'files(id,name,webViewLink)','pageSize':100},timeout=30)
    response.raise_for_status()
    for item in sorted(response.json().get('files',[]),key=lambda x:x['name']):
        if item.get('webViewLink') and item['name'].endswith(('.csv','.txt')):
            display(Markdown(f"[{item['name']}]({item['webViewLink']})"))
except Exception as error:
    print('Drive link lookup:',error)
    print('The CSV files are saved. Search Drive for the folder word_label_overview, or use the Colab file browser:',OUT)
